In [16]:
import cv2
import pytesseract
import numpy as np
from PIL import Image

# Set the path to Tesseract OCR (change this according to your installation)
pytesseract.pytesseract.tesseract_cmd = r"C:\ProgramData\Microsoft\Windows\start menu\Programs\Tesseract-OCR"  # Windows example
# For Linux/Mac: typically '/usr/bin/tesseract'

def preprocess_image(img):
    """Preprocess the image for better OCR results"""
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Apply bilateral filter to reduce noise while keeping edges sharp
    gray = cv2.bilateralFilter(gray, 11, 17, 17)
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                 cv2.THRESH_BINARY, 11, 2)
    
    return thresh

def detect_license_plate(frame):
    """Detect and extract license plate from a frame"""
    # Convert frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Apply filters and edge detection
    blurred = cv2.bilateralFilter(gray, 11, 17, 17)
    edged = cv2.Canny(blurred, 30, 200)
    
    # Find contours
    contours, _ = cv2.findContours(edged.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    
    # Sort contours by area and keep the largest ones
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]
    
    license_plate = None
    plate_contour = None
    
    # Find rectangular contours that could be license plates
    for contour in contours:
        perimeter = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.018 * perimeter, True)
        
        if len(approx) == 4:  # If the contour has 4 vertices
            plate_contour = approx
            x, y, w, h = cv2.boundingRect(contour)
            license_plate = frame[y:y+h, x:x+w]
            break
    
    if license_plate is not None:
        # Preprocess the license plate image
        license_plate = preprocess_image(license_plate)
        
        # Perform OCR
        text = pytesseract.image_to_string(license_plate, 
                                         config='--psm 11 --oem 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
        
        # Clean up the detected text
        text = ''.join(e for e in text if e.isalnum()).upper()
        
        if len(text) > 3:  # Minimum length for a license plate
            # Draw rectangle around license plate
            cv2.drawContours(frame, [plate_contour], -1, (0, 255, 0), 3)
            
            # Put text on the frame
            cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 
                       1, (0, 255, 0), 2, cv2.LINE_AA)
            
            return frame, text
    
    return frame, None

def process_video(video_path, output_path=None):
    """Process a video file for license plate recognition"""
    cap = cv2.VideoCapture(video_path)
    
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        out = cv2.VideoWriter(output_path, fourcc, 20.0, 
                             (int(cap.get(3)), int(cap.get(4))))
    
    plates_detected = set()
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        processed_frame, plate_text = detect_license_plate(frame)
        
        if plate_text:
            plates_detected.add(plate_text)
            print(f"Detected plate: {plate_text}")
        
        if output_path:
            out.write(processed_frame)
        
        cv2.imshow('License Plate Recognition', processed_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    if output_path:
        out.release()
    cv2.destroyAllWindows()
    
    print("\nAll detected license plates:")
    for plate in plates_detected:
        print(plate)

# Example usage
if __name__ == "__main__":
    video_file = "car_video.mp4"  # Replace with your video file path
    output_file = "output_video.avi"  # Optional: to save the processed video
    
    process_video(video_file, output_file)


All detected license plates:
